# Taiwan Literacy Rate & Educational Dynamics (2019–2026)
### Combining Official Demographic Data (MOI/MOE) with Micro Student Study Habits Modeling (Kaggle)

This notebook conducts a data science investigation into:
1. **Macro Trends**: Official census trends of Taiwan's literacy rate for the population aged 15+ (2019–2024) and statistical projections (2025–2026).
2. **Demographics**: Gender parity gap compression, elderly cohort catch-up, and regional performance across Taiwan's 22 counties and cities.
3. **Micro Determinants**: Machine learning modeling on the Kaggle [Student Study Habits](https://www.kaggle.com/datasets/prekshad2166/student-study-habits) dataset to examine how study hours, sleep, attendance, screen time, and cram school tutoring impact literacy mastery.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

DATA_DIR = os.path.join("..", "data", "processed")
print("Libraries loaded successfully.")

## 1. Macro Analysis: Taiwan National Literacy Timeseries (2019–2026)

In [ ]:
national_df = pd.read_csv(os.path.join(DATA_DIR, "taiwan_literacy_national_2019_2026.csv"))
display(national_df)

### Key Observations:
- **Overall Literacy Rate**: Rose from 99.08% in 2019 to 99.32% in 2024, projected to reach 99.47% by 2026.
- **Gender Gap**: Shrunk from 1.26 percentage points in 2019 down to 1.05 in 2024, and projected to narrow further to 0.86 by 2026.
- **Literate Population**: Expanded by over 150,000 citizens to over 20.51 million in 2024.

## 2. Statistical Forecasting: Asymptotic Logistic Decay vs. Linear Model

In [ ]:
def asymptotic_decay(t, i0, k):
    return i0 * np.exp(-k * t)

hist = national_df[national_df["status"] == "Official Historical"]
t_hist = hist["year"].values - 2019
illit_hist = 100.0 - hist["overall_literacy_rate"].values

# Fit Asymptotic Logistic Decay
popt, _ = curve_fit(asymptotic_decay, t_hist, illit_hist, p0=[illit_hist[0], 0.08])
t_proj = np.array([2025 - 2019, 2026 - 2019])
pred_lit_asymp = 100.0 - asymptotic_decay(t_proj, *popt)

print(f"Asymptotic Decay 2025 Forecast: {pred_lit_asymp[0]:.3f}%")
print(f"Asymptotic Decay 2026 Forecast: {pred_lit_asymp[1]:.3f}%")

## 3. Regional Analysis: All 22 Cities & Counties

In [ ]:
counties_df = pd.read_csv(os.path.join(DATA_DIR, "taiwan_literacy_counties_2019_2026.csv"))
# Top 5 and Bottom 5 in 2024
c_2024 = counties_df[counties_df["year"] == 2024].sort_values(by="literacy_rate", ascending=False)
print("Top 5 Regions (2024):")
display(c_2024[["county_en", "county_zh", "category", "literacy_rate"]].head(5))
print("Catch-up Regions (2024):")
display(c_2024[["county_en", "county_zh", "category", "literacy_rate"]].tail(5))

## 4. Micro Student Study Habits Analysis (Kaggle Dataset)

In [ ]:
student_df = pd.read_csv(os.path.join(DATA_DIR, "student_study_habits_processed.csv"))
display(student_df.head(5))

# Correlation matrix
corr = student_df[["study_hours_per_week", "sleep_hours_per_day", "attendance_percentage", "social_media_hours_per_day", "academic_exam_score"]].corr()
print("Correlation with Academic Exam Score:")
display(corr["academic_exam_score"].sort_values(ascending=False))

In [ ]:
# Train Random Forest & Linear Regression on Student Habits
X = student_df[["study_hours_per_week", "sleep_hours_per_day", "attendance_percentage", "social_media_hours_per_day"]].copy()
y = student_df["academic_exam_score"]

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

feature_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Random Forest Feature Importances:")
print(feature_imp)

## 5. Summary & Strategic Insights
1. **Macro Literacy Achievement**: Taiwan maintains virtually complete literacy (>99.3%) among the general population, with the small remaining illiterate segment disappearing as historical cohorts age and community adult education classes continue.
2. **Micro Educational Success**: Weekly study hours and class attendance are the two decisive levers of academic mastery, while excessive screen time presents the largest single behavioral risk factor.